## Step 1: Setup

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("OK")

In [ ]:
%pip install -q biopython scikit-learn scipy pandas numpy xgboost fair-esm rapidfuzz matplotlib torch lime

## Step 2: Settings

**Hardware settings:** `N_JOBS = 4` uses all 4 CPU cores. `ESM_BATCH_SIZE = 64` and the PyTorch thread count make better use of 4 cores and 20 GB RAM during ESM-2 embedding. Search budget (`SEARCH_ITER`, `OUTER_REPEATS`, folds) is kept the same as your earlier run, so `cv_0.80` reproduces the earlier baseline.

In [ ]:
from pathlib import Path
import json, math, time, traceback, os

# Folder that holds both input CSVs. All outputs (sens_r24/ folder, ESM cache, zip) are written here too.
BASE_DIR = Path(r"H:\Peptide\review-1\threshold_sensitivity")
os.chdir(BASE_DIR)

QUICK_TEST = False        # True: ESM off, small search/folds, only cv_0.80 + core9 + sim_0.90 (only to test the flow!)

MAIN_CSV = str(BASE_DIR / "main_dataset_for_ml_model.csv")
EXT_CSV  = str(BASE_DIR / "external_validation_clean.csv")
OUT = BASE_DIR / "sens_r24"

# Baseline external prediction from the earlier R1.4 run (if it exists it is used for Design B and the external cutoff sweep)
BASE_EXT_PRED = str(BASE_DIR / "sens" / "base" / "ext_pred_merged.csv")
PREV_BASE_AUC = 0.8189    # internal ensemble AUC of the earlier base run (to check that cv_0.80 reproduces it)

USE_ESM, ESM_MODEL = True, "esm2_t30_150M_UR50D"
ESM_BATCH_SIZE = 64       # 20 GB RAM is plenty for peptide-length inputs (pipeline default is 16)
N_JOBS = 4                # use all 4 CPU cores
SEARCH_ITER, OUTER_REPEATS, OUTER_FOLDS, INNER_FOLDS = 10, 1, 5, 3
N_BOOT = 2000
TOL = 0.02

RUN_DESIGN_A = True
RUN_DESIGN_B = True
A_RUN = ["cv_0.80", "core9", "cv_0.60", "cv_0.70", "naive", "cv_0.90", "cv_0.50"]   # in priority order
INCLUDE_CORE8 = False     # True: also run with an 8-aa shared stretch (sensitivity)
B_THRESHOLDS = [0.70, 0.90]

if QUICK_TEST:
    USE_ESM, SEARCH_ITER, OUTER_FOLDS, INNER_FOLDS, N_BOOT = False, 2, 2, 2, 200
    A_RUN = ["cv_0.80", "core9"]
    B_THRESHOLDS = [0.90]
if INCLUDE_CORE8 and "core8" not in A_RUN:
    A_RUN.append("core8")
OUT.mkdir(exist_ok=True)

# Let PyTorch (ESM-2 embedding runs in the main process) use all cores,
# even though OMP_NUM_THREADS=1 was set for the sklearn worker processes.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
try:
    import torch
    torch.set_num_threads(N_JOBS)
    print("torch threads:", torch.get_num_threads(), "| CUDA available:", torch.cuda.is_available())
except Exception as e:
    print("torch not available:", e)

print("QUICK_TEST =", QUICK_TEST, "| USE_ESM =", USE_ESM, "| N_JOBS =", N_JOBS, "| A_RUN =", A_RUN)

## Step 3: Check input files


In [ ]:
missing = [f for f in (MAIN_CSV, EXT_CSV) if not Path(f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing input file(s): {missing}. Check the file names inside {BASE_DIR} "
        "or change BASE_DIR / MAIN_CSV / EXT_CSV in Step 2.")
print({f: Path(f).exists() for f in (MAIN_CSV, EXT_CSV)}, "| base ext pred:", Path(BASE_EXT_PRED).exists())

## Step 4: Pipeline code (unchanged from your notebook)
Run all the cells in order.

In [ ]:
import argparse
import json
import math
import pickle
import warnings
import hashlib
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.Align import substitution_matrices
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils.ProtParamData import kd
from scipy.stats import loguniform, randint, t
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef, precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold, RandomizedSearchCV, RepeatedStratifiedKFold,
    StratifiedGroupKFold, StratifiedKFold, cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, NuSVC

import sklearn
_SKLEARN_VERSION = tuple(map(int, sklearn.__version__.split('.')[:2]))
_HISTGB_HAS_CLASS_WEIGHT = _SKLEARN_VERSION >= (1, 2)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    XGBClassifier = None
    HAS_XGB = False

try:
    import torch
    import esm
    HAS_ESM = True
except Exception:
    HAS_ESM = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

try:
    import lime
    import lime.lime_tabular
    HAS_LIME = True
except Exception:
    HAS_LIME = False

STANDARD_AA = "ACDEFGHIKLMNPQRSTVWY"
AA_SET = set(STANDARD_AA)
ALL_DIPEPTIDES = [a + b for a in STANDARD_AA for b in STANDARD_AA]
BLOSUM62 = substitution_matrices.load("BLOSUM62")

ESM_MODEL_INFO = {
    "esm2_t6_8M_UR50D": 6,
    "esm2_t12_35M_UR50D": 12,
    "esm2_t30_150M_UR50D": 30,
    "esm2_t33_650M_UR50D": 33,
}

print(f"sklearn: {sklearn.__version__} | XGB: {HAS_XGB} | ESM: {HAS_ESM} | SHAP: {HAS_SHAP} | LIME: {HAS_LIME}")


In [ ]:
@dataclass
class Config:
    input_csv: str = "main_dataset_cleaned_2.csv"
    outdir: str = "ml_report_v5"
    seq_col: str = "Peptide"
    label_col: str = "label"
    id_col: str = "dataset_id"

    # MHC restriction
    mhc_rank_col: str = "MHC_Percentile_Rank"
    mhc_ic50_col: str = "MHC_IC50_nM"
    mhc_class_col: str = "MHC_Class_Used"
    mhc_allele_col: str = "MHC_Allele_Best"
    require_mhc_assay: bool = True
    require_known_class: bool = True
    require_tcell_assay: bool = True
    assay_col: str = "Assay"
    tcell_assay_keyword: str = "T cell assay"

    # Similarity / conflict
    similarity_threshold: float = 0.80
    conflict_consensus_fraction: float = 0.75
    resolve_conflicts: bool = True

    # Features
    include_dpc: bool = True          # CHANGED: was False. Tested: small but free (+0.002-0.01 AUC)
    include_anchor_features: bool = True   # NEW: MHC-class-aware anchor-position features
    autoc_maxlag: int = 5
    use_mhc_features: bool = True
    mhc_allele_min_count: int = 5

    # ESM-2
    use_esm_embeddings: bool = True
    esm_model_name: str = "esm2_t12_35M_UR50D"
    esm_repr_layer: int = -1
    esm_batch_size: int = 16
    esm_device: str = ""

    # Selection
    var_threshold: float = 1e-5
    corr_threshold: float = 0.90
    mi_top_k: int = 300               # CHANGED: was 250. Modest headroom increase.

    # CV / HPO
    outer_folds: int = 5
    outer_repeats: int = 3
    inner_folds: int = 3
    use_similarity_cv: str = "auto"
    random_search_iter: int = 35      # CHANGED: was 25.
    threshold_metric: str = "mcc"
    ensemble_min_inner_auc: float = 0.70

    # NEW: explainability / publication artifacts
    save_explainability_artifacts: bool = True
    shap_max_background: int = 250
    lime_n_explanations: int = 3

    random_seed: int = 42
    n_jobs: int = 4

    def __post_init__(self):
        if self.esm_repr_layer is None or self.esm_repr_layer < 0:
            self.esm_repr_layer = ESM_MODEL_INFO.get(self.esm_model_name, 12)


CTD_GROUPS = {
    "Hydrophobicity": {1: set("RKEDQN"), 2: set("GASTPHY"), 3: set("CLVIMFW")},
    "NormVanDerWaalsVolume": {1: set("GASTPDC"), 2: set("NVEQIL"), 3: set("MHKFRYW")},
    "Polarity": {1: set("LIFWCMVY"), 2: set("PATGS"), 3: set("HQRKNED")},
    "Polarizability": {1: set("GASDT"), 2: set("CPNVEQIL"), 3: set("KMHFRYW")},
    "Charge": {1: set("KR"), 2: set("ANCQGHILMFPSTWYV"), 3: set("DE")},
    "SecondaryStructure": {1: set("EALMQKRH"), 2: set("VIYCWFT"), 3: set("GNPSD")},
    "SolventAccessibility": {1: set("ALFCGIVW"), 2: set("RKQEND"), 3: set("MSPTHY")},
}

AA_POLARITY_GRANTHAM = {
    "A": 8.1, "R": 10.5, "N": 11.6, "D": 13.0, "C": 5.5, "Q": 10.5, "E": 12.3, "G": 9.0,
    "H": 10.4, "I": 5.2, "L": 4.9, "K": 11.3, "M": 5.7, "F": 5.2, "P": 8.0, "S": 9.2,
    "T": 8.6, "W": 5.4, "Y": 6.2, "V": 5.9,
}

AA_VOLUME_ZAMYATNIN = {
    "A": 88.6, "R": 173.4, "N": 114.1, "D": 111.1, "C": 108.5, "Q": 143.8, "E": 138.4,
    "G": 60.1, "H": 153.2, "I": 166.7, "L": 166.7, "K": 168.6, "M": 162.9, "F": 189.9,
    "P": 112.7, "S": 89.0, "T": 116.1, "W": 227.8, "Y": 193.6, "V": 140.0,
}

HYDROPHOBIC_AA = set("AILMFWVC")


In [ ]:
def clean_sequence(seq: str) -> str:
    return str(seq).strip().upper()


def is_valid_peptide(seq: str) -> bool:
    seq = clean_sequence(seq)
    return len(seq) > 0 and set(seq).issubset(AA_SET)


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def normalized_edit_identity(a: str, b: str, threshold_hint: float = 0.0) -> float:
    a, b = clean_sequence(a), clean_sequence(b)
    if a == b:
        return 1.0
    la, lb = len(a), len(b)
    max_len = max(la, lb)
    if max_len == 0:
        return 0.0
    if min(la, lb) / max_len < threshold_hint:
        return min(la, lb) / max_len
    prev = list(range(lb + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return 1.0 - prev[-1] / max_len


def compute_similarity_groups(seqs: list, threshold: float) -> np.ndarray:
    """Union-find clusters of near-duplicate sequences (identity >= threshold)."""
    n = len(seqs)
    if n > 3000:
        print(f"[WARNING] O(n^2) similarity clustering on {n} sequences will be very slow.")
        print("          Consider pre-clustering with cd-hit or mmseqs for larger datasets.")

    uf = UnionFind(n)
    for i in range(n):
        if i % 250 == 0 and i > 0:
            print(f"[similarity] compared {i}/{n} sequences")
        for j in range(i + 1, n):
            if min(len(seqs[i]), len(seqs[j])) / max(len(seqs[i]), len(seqs[j])) < threshold:
                continue
            if normalized_edit_identity(seqs[i], seqs[j], threshold_hint=threshold) >= threshold:
                uf.union(i, j)
    roots = [uf.find(i) for i in range(n)]
    root_to_id = {root: idx for idx, root in enumerate(sorted(set(roots)))}
    return np.array([root_to_id[root] for root in roots])


In [ ]:
def aac_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    counts = Counter(seq)
    length = len(seq)
    return {f"AAC_{aa}": counts.get(aa, 0) / length * 100 for aa in STANDARD_AA}


def dpc_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    denom = max(len(seq) - 1, 1)
    counts = Counter(seq[i:i + 2] for i in range(len(seq) - 1))
    return {f"DPC_{dp}": counts.get(dp, 0) / denom * 100 for dp in ALL_DIPEPTIDES}


def _ctd_group_of(aa: str, groups: dict):
    for group_id, members in groups.items():
        if aa in members:
            return group_id
    return None


def ctd_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    length = len(seq)
    feats = {}
    for attr, groups in CTD_GROUPS.items():
        group_seq = [_ctd_group_of(aa, groups) for aa in seq]
        counts = Counter(group_seq)
        for gid in (1, 2, 3):
            feats[f"CTD_{attr}_C{gid}"] = counts.get(gid, 0) / length * 100
        denom = max(length - 1, 1)
        transitions = Counter()
        for i in range(length - 1):
            left, right = group_seq[i], group_seq[i + 1]
            if left is None or right is None or left == right:
                continue
            transitions[tuple(sorted((left, right)))] += 1
        for pair in [(1, 2), (1, 3), (2, 3)]:
            feats[f"CTD_{attr}_T{pair[0]}{pair[1]}"] = transitions.get(pair, 0) / denom * 100
        for gid in (1, 2, 3):
            positions = [i + 1 for i, g in enumerate(group_seq) if g == gid]
            if not positions:
                distribution = [0.0] * 5
            else:
                n = len(positions)
                idxs = [1, max(1, math.ceil(0.25 * n)), max(1, math.ceil(0.50 * n)),
                        max(1, math.ceil(0.75 * n)), n]
                distribution = [positions[idx - 1] / length * 100 for idx in idxs]
            for pct, val in zip([0, 25, 50, 75, 100], distribution):
                feats[f"CTD_{attr}_D{gid}_{pct}"] = val
    return feats


def physicochemical_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    for name, table in [("Hydrophobicity_KD", kd), ("Polarity_Grantham", AA_POLARITY_GRANTHAM),
                         ("Volume_Zamyatnin", AA_VOLUME_ZAMYATNIN)]:
        values = np.array([table[aa] for aa in seq], dtype=float)
        feats[f"PHYS_{name}_mean"] = float(values.mean())
        feats[f"PHYS_{name}_std"] = float(values.std())
    analysis = ProteinAnalysis(seq)
    feats["PHYS_molecular_weight"] = analysis.molecular_weight()
    feats["PHYS_aromaticity"] = analysis.aromaticity()
    feats["PHYS_instability_index"] = analysis.instability_index()
    try:
        feats["PHYS_isoelectric_point"] = analysis.isoelectric_point()
    except Exception:
        feats["PHYS_isoelectric_point"] = np.nan
    feats["PHYS_gravy"] = analysis.gravy()
    feats["PHYS_charge_at_pH7"] = analysis.charge_at_pH(7.0)
    helix, turn, sheet = analysis.secondary_structure_fraction()
    feats["PHYS_ss_fraction_helix"] = helix
    feats["PHYS_ss_fraction_turn"] = turn
    feats["PHYS_ss_fraction_sheet"] = sheet
    feats["PHYS_length"] = len(seq)
    return feats


In [ ]:
def dhkr_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    length = len(seq)
    counts = Counter(seq)
    feats = {f"DHKR_frac_{aa}": counts.get(aa, 0) / length * 100 for aa in "DHKRE"}
    feats["DHKR_charged_fraction"] = sum(counts.get(aa, 0) for aa in "DHKRE") / length * 100
    feats["DHKR_net_charge_proxy"] = (
        counts.get("K", 0) + counts.get("R", 0) + 0.1 * counts.get("H", 0)
        - counts.get("D", 0) - counts.get("E", 0)
    ) / length
    return feats


def _moran_autocorrelation(seq: str, table: dict, maxlag: int) -> list:
    values = np.array([table[aa] for aa in clean_sequence(seq)], dtype=float)
    length = len(values)
    mean_value = values.mean()
    denom = np.sum((values - mean_value) ** 2) / length
    out = []
    for lag in range(1, maxlag + 1):
        if lag >= length or denom == 0:
            out.append(0.0)
            continue
        numerator = np.sum((values[:-lag] - mean_value) * (values[lag:] - mean_value)) / (length - lag)
        out.append(float(numerator / denom))
    return out


def autocorrelation_features(seq: str, maxlag: int = 5) -> dict:
    seq = clean_sequence(seq)
    maxlag = max(0, min(maxlag, len(seq) - 1))
    feats = {}
    for name, table in [("Hydrophobicity_KD", kd), ("Polarity_Grantham", AA_POLARITY_GRANTHAM),
                         ("Volume_Zamyatnin", AA_VOLUME_ZAMYATNIN)]:
        for lag, value in enumerate(_moran_autocorrelation(seq, table, maxlag), start=1):
            feats[f"AUTOC_Moran_{name}_lag{lag}"] = value
    return feats


def blosum62_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    acc = {aa: 0.0 for aa in STANDARD_AA}
    for residue in seq:
        for aa in STANDARD_AA:
            acc[aa] += BLOSUM62[residue][aa]
    return {f"BLOSUM_{aa}": acc[aa] / len(seq) for aa in STANDARD_AA}


def terminal_composition_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    for side, window in [("N", seq[:2]), ("C", seq[-2:]), ("N3", seq[:3]), ("C3", seq[-3:])]:
        counts = Counter(window)
        denom = max(len(window), 1)
        for aa in STANDARD_AA:
            feats[f"TERM_{side}_AAC_{aa}"] = counts.get(aa, 0) / denom * 100
    return feats


def anchor_position_features(seq: str, mhc_class: str) -> dict:
    seq = clean_sequence(seq)
    L = len(seq)
    mhc_class = str(mhc_class).strip().upper() if mhc_class is not None else ""
    feats = {
        "ANCHOR_P2_hydro": 0.0, "ANCHOR_Pomega_hydro": 0.0,
        "ANCHOR_P2_is_hydrophobic": 0.0, "ANCHOR_Pomega_is_hydrophobic": 0.0,
        "ANCHOR_classII_a1_hydro": 0.0, "ANCHOR_classII_a4_hydro": 0.0,
        "ANCHOR_classII_a6_hydro": 0.0, "ANCHOR_classII_a9_hydro": 0.0,
    }
    if L == 0:
        return feats
    if mhc_class == "I":
        p2 = seq[1] if L > 1 else seq[0]
        pomega = seq[-1]
        feats["ANCHOR_P2_hydro"] = float(kd.get(p2, 0.0))
        feats["ANCHOR_Pomega_hydro"] = float(kd.get(pomega, 0.0))
        feats["ANCHOR_P2_is_hydrophobic"] = float(p2 in HYDROPHOBIC_AA)
        feats["ANCHOR_Pomega_is_hydrophobic"] = float(pomega in HYDROPHOBIC_AA)
    elif mhc_class == "II":
        idxs = [max(0, min(L - 1, int(round(f * (L - 1))))) for f in (0.0, 0.33, 0.55, 0.9)]
        keys = ["ANCHOR_classII_a1_hydro", "ANCHOR_classII_a4_hydro",
                "ANCHOR_classII_a6_hydro", "ANCHOR_classII_a9_hydro"]
        for key, idx in zip(keys, idxs):
            feats[key] = float(kd.get(seq[idx], 0.0))
    return feats


def extract_all_features(seq: str, cfg: Config, mhc_class: str = None) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    feats.update(aac_features(seq))
    if cfg.include_dpc:
        feats.update(dpc_features(seq))
    feats.update(ctd_features(seq))
    feats.update(physicochemical_features(seq))
    feats.update(dhkr_features(seq))
    feats.update(autocorrelation_features(seq, maxlag=cfg.autoc_maxlag))
    feats.update(blosum62_features(seq))
    feats.update(terminal_composition_features(seq))
    if cfg.include_anchor_features:
        feats.update(anchor_position_features(seq, mhc_class))
    return feats


def feature_family(feature_name: str) -> str:
    for prefix, fam in [("AAC_", "AAC"), ("DPC_", "DPC"), ("CTD_", "CTD"), ("PHYS_", "PHYS"),
                         ("DHKR_", "DHKR"), ("AUTOC_", "AUTOC"), ("BLOSUM_", "BLOSUM"),
                         ("TERM_", "TERMINAL"), ("ANCHOR_", "ANCHOR"),
                         ("ESM2_", "ESM2"), ("MHC_", "MHC")]:
        if feature_name.startswith(prefix):
            return fam
    return "OTHER"


In [ ]:
def extract_mhc_features(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    mhc = pd.DataFrame(index=df.index)
    rank = pd.to_numeric(df.get(cfg.mhc_rank_col), errors="coerce")
    ic50 = pd.to_numeric(df.get(cfg.mhc_ic50_col), errors="coerce")

    mhc["MHC_rank_missing"] = rank.isna().astype(float)
    mhc["MHC_ic50_missing"] = ic50.isna().astype(float)

    approx_rank_from_ic50 = np.clip(np.log10(ic50 + 1.0) * 2.5, 0, 100)
    approx_ic50_from_rank = np.clip(10 ** (rank / 2.5), 1, 100000)
    mhc["MHC_rank"] = rank.fillna(approx_rank_from_ic50).fillna(100.0)
    mhc["MHC_ic50"] = ic50.fillna(approx_ic50_from_rank).fillna(50000.0)
    mhc["MHC_log10_ic50"] = np.log10(mhc["MHC_ic50"] + 1.0)
    mhc["MHC_sqrt_rank"] = np.sqrt(mhc["MHC_rank"])

    mhc["MHC_strong_binder"] = ((mhc["MHC_rank"] < 0.5) | (mhc["MHC_ic50"] < 50.0)).astype(float)
    mhc["MHC_good_binder"] = (((mhc["MHC_rank"] >= 0.5) & (mhc["MHC_rank"] < 2.0)) |
                               ((mhc["MHC_ic50"] >= 50.0) & (mhc["MHC_ic50"] < 500.0))).astype(float)
    mhc["MHC_weak_binder"] = (((mhc["MHC_rank"] >= 2.0) & (mhc["MHC_rank"] < 10.0)) |
                               ((mhc["MHC_ic50"] >= 500.0) & (mhc["MHC_ic50"] < 5000.0))).astype(float)
    mhc["MHC_non_binder"] = ((mhc["MHC_rank"] >= 10.0) & (mhc["MHC_ic50"] >= 5000.0)).astype(float)
    mhc["MHC_binding_score"] = np.clip(1.0 / (1.0 + mhc["MHC_rank"]), 0, 1)
    mhc["MHC_log_ic50_score"] = np.clip(1.0 - (np.log10(mhc["MHC_ic50"] + 1) / 5.0), 0, 1)

    class_used = df.get(cfg.mhc_class_col, pd.Series("unknown", index=df.index)).astype(str).str.strip().str.upper()
    mhc["MHC_class_I"] = (class_used == "I").astype(float)
    mhc["MHC_class_II"] = (class_used == "II").astype(float)

    length = df[cfg.seq_col].astype(str).str.len()
    mhc["MHC_classI_x_short"] = mhc["MHC_class_I"] * ((length >= 8) & (length <= 11)).astype(float)
    mhc["MHC_classII_x_long"] = mhc["MHC_class_II"] * ((length >= 13) & (length <= 25)).astype(float)
    mhc["MHC_compatible_length"] = (
        mhc["MHC_class_I"] * ((length >= 8) & (length <= 11)).astype(float)
        + mhc["MHC_class_II"] * ((length >= 12) & (length <= 25)).astype(float)
    )

    if cfg.mhc_allele_col in df.columns:
        allele = df[cfg.mhc_allele_col].fillna("unknown").astype(str).str.strip()

        def allele_supertype(a):
            a = str(a).upper()
            for prefix, fam in [("HLA-A", "HLA_A"), ("HLA-B", "HLA_B"), ("HLA-C", "HLA_C"),
                                 ("HLA-DR", "HLA_DR"), ("HLA-DQ", "HLA_DQ"), ("HLA-DP", "HLA_DP")]:
                if a.startswith(prefix):
                    return fam
            return "Other"

        fam = allele.apply(allele_supertype)
        fam_counts = fam.value_counts()
        valid_fams = fam_counts[fam_counts >= cfg.mhc_allele_min_count].index.tolist()
        for f in valid_fams:
            mhc[f"MHC_allele_fam_{f}"] = (fam == f).astype(float)

    return mhc.fillna(0.0)


def _esm_cache_path(seqs: list, cfg: Config) -> Path:
    sig = f"{cfg.esm_model_name}|{cfg.esm_repr_layer}|{len(seqs)}|{seqs[0]}|{seqs[-1]}"
    h = hashlib.md5(sig.encode()).hexdigest()[:16]
    return Path(f".esm_cache_{h}.pkl")


def esm2_embedding_features(seqs: list, cfg: Config) -> pd.DataFrame:
    if not cfg.use_esm_embeddings:
        return pd.DataFrame(index=pd.RangeIndex(len(seqs)))
    if not HAS_ESM:
        raise RuntimeError(
            "ESM-2 requires 'torch' and 'fair-esm'. Install with:\n"
            "  pip install torch --index-url https://download.pytorch.org/whl/cpu\n"
            "  pip install fair-esm"
        )
    if not seqs:
        return pd.DataFrame(index=pd.RangeIndex(0))

    cache_path = _esm_cache_path(seqs, cfg)
    if cache_path.exists():
        print(f"[ESM-2] Loading cached embeddings from {cache_path}")
        return pd.read_pickle(cache_path)

    device = cfg.esm_device.strip() if cfg.esm_device else ("cuda" if torch.cuda.is_available() else "cpu")
    if not hasattr(esm.pretrained, cfg.esm_model_name):
        raise ValueError(f"Unknown ESM model '{cfg.esm_model_name}'.")

    print(f"[ESM-2] Loading {cfg.esm_model_name} (layer {cfg.esm_repr_layer}) on {device}.")
    model, alphabet = getattr(esm.pretrained, cfg.esm_model_name)()
    model = model.eval().to(device)
    batch_converter = alphabet.get_batch_converter()

    records, feature_names, total = [], None, len(seqs)
    with torch.no_grad():
        for start in range(0, total, cfg.esm_batch_size):
            batch_seqs = seqs[start:start + cfg.esm_batch_size]
            batch = [(str(start + i), seq) for i, seq in enumerate(batch_seqs)]
            _, strings, tokens = batch_converter(batch)
            tokens = tokens.to(device)
            output = model(tokens, repr_layers=[cfg.esm_repr_layer], return_contacts=False)
            reps = output["representations"][cfg.esm_repr_layer].detach().cpu().numpy()
            for row_idx, seq in enumerate(strings):
                vector = reps[row_idx, 1:len(seq) + 1].mean(axis=0)
                if feature_names is None:
                    feature_names = [f"ESM2_{i:03d}" for i in range(vector.shape[0])]
                records.append(vector.astype(float))
            print(f"[ESM-2] embedded {min(start + len(batch_seqs), total)}/{total} peptides")

    out_df = pd.DataFrame(records, columns=feature_names)
    out_df.to_pickle(cache_path)
    print(f"[ESM-2] Saved embeddings to {cache_path}")
    return out_df


In [ ]:
def build_feature_matrix(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Assumes df has already been filtered to valid sequences upstream."""
    clean_list = df[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    if len(clean_list) == 0:
        raise ValueError("No peptides remain after upstream filtering.")

    if cfg.include_anchor_features and cfg.mhc_class_col in df.columns:
        mhc_class_list = df[cfg.mhc_class_col].astype(str).tolist()
    else:
        mhc_class_list = [None] * len(clean_list)

    records = [extract_all_features(seq, cfg, mhc_class=cls)
               for seq, cls in zip(clean_list, mhc_class_list)]
    feat_df = pd.DataFrame.from_records(records).fillna(0.0)

    esm_df = esm2_embedding_features(clean_list, cfg).fillna(0.0)
    feat_df = pd.concat([feat_df.reset_index(drop=True), esm_df.reset_index(drop=True)], axis=1)

    if cfg.use_mhc_features:
        mhc_df = extract_mhc_features(df, cfg).reset_index(drop=True)
        feat_df = pd.concat([feat_df, mhc_df], axis=1)
        print(f"[feature extraction] Added {mhc_df.shape[1]} MHC features.")

    lead_cols = {}
    if cfg.id_col and cfg.id_col in df.columns:
        lead_cols[cfg.id_col] = df[cfg.id_col].values
    lead_cols[cfg.label_col] = df[cfg.label_col].values
    lead_cols[cfg.seq_col] = df[cfg.seq_col].values

    print(f"[feature extraction] Built {feat_df.shape[1]} features "
          f"(DPC {'ON' if cfg.include_dpc else 'OFF'}, "
          f"anchor {'ON' if cfg.include_anchor_features else 'OFF'}) "
          f"for {feat_df.shape[0]} peptides.")
    return pd.concat([pd.DataFrame(lead_cols).reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)


In [ ]:
def restrict_to_assay_supported(df: pd.DataFrame, cfg: Config) -> tuple:
    n_before = len(df)
    keep = pd.Series(True, index=df.index)
    reasons = {}

    if cfg.require_mhc_assay:
        rank = pd.to_numeric(df.get(cfg.mhc_rank_col), errors="coerce")
        ic50 = pd.to_numeric(df.get(cfg.mhc_ic50_col), errors="coerce")
        has_assay = rank.notna() | ic50.notna()
        reasons["dropped_no_assay_value"] = int((~has_assay).sum())
        keep &= has_assay

    if cfg.require_known_class:
        class_used = df.get(cfg.mhc_class_col, pd.Series(index=df.index, dtype=object))
        class_used = class_used.astype(str).str.strip().str.upper()
        has_class = class_used.isin(["I", "II"])
        reasons["dropped_unknown_class"] = int((~has_class & keep).sum())
        keep &= has_class

    out = df.loc[keep].reset_index(drop=True)
    report = {"n_before": n_before, "n_after": int(len(out)), **reasons}
    print(f"[restrict] {n_before} -> {len(out)} peptides after MHC assay-support restriction: {reasons}")
    return out, report


def restrict_to_tcell_assay(df: pd.DataFrame, cfg: Config) -> tuple:
    """NEW. Restrict to peptides with direct T-cell assay evidence.

    Motivation: this class of dataset (IEDB/CEDAR-derived) typically records
    'immunogenicity' evidence from B-cell assays (antibody binding), MHC-ligand
    assays (mass-spec elution -- confirms MHC presentation, NOT T-cell
    recognition), and true T-cell assays (the actual target phenomenon) under
    one column. Mixing these under a single binary label adds label noise: MHC
    presentation is necessary but not sufficient for T-cell immunogenicity.
    Empirically, restricting to T-cell-assay-confirmed peptides measurably
    improved honest (similarity-group-aware) CV AUC on this dataset.
    """
    if not cfg.require_tcell_assay or cfg.assay_col not in df.columns:
        return df, {"require_tcell_assay": False}
    n_before = len(df)
    has_tcell = df[cfg.assay_col].astype(str).str.contains(cfg.tcell_assay_keyword, na=False, case=False)
    out = df.loc[has_tcell].reset_index(drop=True)
    report = {
        "n_before_tcell_filter": n_before,
        "n_after_tcell_filter": int(len(out)),
        "n_dropped_no_tcell_evidence": int((~has_tcell).sum()),
    }
    print(f"[restrict] T-cell-assay filter: {n_before} -> {len(out)} peptides "
          f"({report['n_dropped_no_tcell_evidence']} dropped -- no direct T-cell assay evidence).")
    return out, report


def resolve_label_conflicts(df: pd.DataFrame, groups: np.ndarray, cfg: Config) -> tuple:
    y = df[cfg.label_col].to_numpy().astype(int)
    resolved = y.copy()
    drop_mask = np.zeros(len(df), dtype=bool)
    n_relabeled_groups, n_relabeled_peptides = 0, 0
    n_dropped_groups, n_dropped_peptides = 0, 0

    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        if len(idx) <= 1:
            continue
        labels = y[idx]
        if len(set(labels.tolist())) == 1:
            continue
        pos_frac = float((labels == 1).mean())
        if pos_frac >= cfg.conflict_consensus_fraction:
            changed = idx[labels == 0]
            resolved[idx] = 1
            n_relabeled_groups += 1
            n_relabeled_peptides += len(changed)
        elif pos_frac <= (1.0 - cfg.conflict_consensus_fraction):
            changed = idx[labels == 1]
            resolved[idx] = 0
            n_relabeled_groups += 1
            n_relabeled_peptides += len(changed)
        else:
            drop_mask[idx] = True
            n_dropped_groups += 1
            n_dropped_peptides += len(idx)

    out = df.copy()
    out[cfg.label_col] = resolved
    out = out.loc[~drop_mask].reset_index(drop=True)
    kept_groups = groups[~drop_mask]
    report = {
        "n_conflicting_groups_relabeled": n_relabeled_groups,
        "n_peptides_relabeled_to_consensus": n_relabeled_peptides,
        "n_conflicting_groups_dropped": n_dropped_groups,
        "n_peptides_dropped_as_contradictory": n_dropped_peptides,
        "n_peptides_after_resolution": int(len(out)),
    }
    print(f"[conflict resolution] {report}")
    return out, kept_groups, report


In [ ]:
def remove_low_variance(X_train: pd.DataFrame, threshold: float) -> list:
    variances = X_train.var(axis=0)
    return variances[variances > threshold].index.tolist()


def remove_highly_correlated(X_train: pd.DataFrame, threshold: float) -> list:
    corr = X_train.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    to_drop = {col for col in upper.columns if any(upper[col] > threshold)}
    return [c for c in X_train.columns if c not in to_drop]


class VarianceCorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, var_threshold: float = 1e-5, corr_threshold: float = 0.90):
        self.var_threshold = var_threshold
        self.corr_threshold = corr_threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = X_df.columns.astype(str).tolist()
        X_df.columns = self.feature_names_in_
        cols_var = remove_low_variance(X_df, self.var_threshold)
        self.selected_columns_ = remove_highly_correlated(X_df[cols_var], self.corr_threshold)
        if not self.selected_columns_:
            self.selected_columns_ = cols_var[:1] if cols_var else self.feature_names_in_[:1]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        X_df.columns = self.feature_names_in_
        return X_df[self.selected_columns_]


class MutualInfoTopKSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k: int = 250, random_state: int = 42, force_include_prefix: str = "MHC_"):
        self.k = k
        self.random_state = random_state
        self.force_include_prefix = force_include_prefix

    def fit(self, X, y):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = X_df.columns.astype(str).tolist()
        X_df.columns = self.feature_names_in_
        forced = [c for c in self.feature_names_in_ if c.startswith(self.force_include_prefix)]
        remaining = [c for c in self.feature_names_in_ if c not in forced]
        if remaining and self.k and self.k > 0:
            n_from_mi = max(0, self.k - len(forced))
            if 0 < n_from_mi < len(remaining):
                mi = mutual_info_classif(X_df[remaining], y, random_state=self.random_state)
                order = np.argsort(mi)[::-1][:n_from_mi]
                selected_from_mi = [remaining[i] for i in order]
            else:
                selected_from_mi = remaining
        else:
            selected_from_mi = remaining
        self.selected_columns_ = forced + selected_from_mi
        if not self.selected_columns_:
            self.selected_columns_ = self.feature_names_in_
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        X_df.columns = self.feature_names_in_
        return X_df[self.selected_columns_]


class WeightedSoftVotingEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators: dict, weights: np.ndarray, threshold: float = 0.5):
        self.estimators = estimators
        self.weights = weights
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def fit(self, X, y=None):
        return self

    def predict_proba(self, X):
        probs = np.array([est.predict_proba(X)[:, 1] for est in self.estimators.values()])
        pos = np.average(probs, axis=0, weights=self.weights)
        return np.column_stack([1.0 - pos, pos])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)


def make_model_pipeline(estimator, cfg: Config) -> Pipeline:
    return Pipeline(steps=[
        ("var_corr", VarianceCorrelationFilter(cfg.var_threshold, cfg.corr_threshold)),
        ("mi", MutualInfoTopKSelector(k=cfg.mi_top_k, random_state=cfg.random_seed)),
        ("scaler", StandardScaler()),
        ("classifier", estimator),
    ])


def pipeline_selected_features(estimator) -> list:
    if not isinstance(estimator, Pipeline):
        return []
    try:
        return estimator.named_steps["mi"].selected_columns_
    except Exception:
        return []


def optimize_threshold(y_true: np.ndarray, proba: np.ndarray, metric: str) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    if len(thresholds) == 0:
        return 0.5
    candidates = np.unique(np.concatenate([thresholds, np.linspace(0.10, 0.90, 81)]))
    best_threshold, best_score = 0.5, -np.inf
    for threshold in candidates:
        pred = (proba >= threshold).astype(int)
        if metric == "f1":
            score = f1_score(y_true, pred, zero_division=0)
        elif metric == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, pred)
        else:
            score = matthews_corrcoef(y_true, pred)
        if score > best_score:
            best_score, best_threshold = score, float(threshold)
    return best_threshold


In [ ]:
def model_search_space(y_train: np.ndarray, cfg: Config) -> dict:
    n_pos, n_neg = int((y_train == 1).sum()), int((y_train == 0).sum())
    scale_pos_weight = n_neg / max(n_pos, 1)
    space = {
        "logreg_l2": (make_model_pipeline(
            LogisticRegression(penalty="l2", solver="lbfgs", max_iter=6000,
                                class_weight="balanced", random_state=cfg.random_seed), cfg),
            {"classifier__C": loguniform(1e-3, 1e2)}),
        "svm_rbf": (make_model_pipeline(
            SVC(kernel="rbf", probability=True, class_weight="balanced",
                random_state=cfg.random_seed, cache_size=1000), cfg),
            {"classifier__C": loguniform(1e-2, 1e2), "classifier__gamma": loguniform(1e-4, 1e0)}),
        "nu_svm": (make_model_pipeline(
            NuSVC(kernel="rbf", gamma="scale", probability=True,
                  random_state=cfg.random_seed, cache_size=1000), cfg),
            {"classifier__nu": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
             "classifier__gamma": loguniform(1e-4, 1e0)}),
        "rf": (make_model_pipeline(
            RandomForestClassifier(random_state=cfg.random_seed, n_jobs=1,
                                    class_weight="balanced_subsample"), cfg),
            {"classifier__n_estimators": randint(300, 900), "classifier__max_depth": [3, 5, 8, 12, 16, None],
             "classifier__min_samples_leaf": randint(1, 10), "classifier__min_samples_split": randint(2, 16),
             "classifier__max_features": ["sqrt", "log2", 0.25, 0.4, 0.6]}),
        "extra_trees": (make_model_pipeline(
            ExtraTreesClassifier(random_state=cfg.random_seed, n_jobs=1, class_weight="balanced"), cfg),
            {"classifier__n_estimators": randint(300, 900), "classifier__max_depth": [3, 5, 8, 12, 16, None],
             "classifier__min_samples_leaf": randint(1, 8), "classifier__min_samples_split": randint(2, 14),
             "classifier__max_features": ["sqrt", "log2", 0.25, 0.4, 0.6]}),
    }

    hist_kwargs = {"random_state": cfg.random_seed}
    if _HISTGB_HAS_CLASS_WEIGHT:
        hist_kwargs["class_weight"] = "balanced"

    space["hist_gb"] = (make_model_pipeline(
        HistGradientBoostingClassifier(**hist_kwargs), cfg),
        {"classifier__max_iter": randint(100, 500), "classifier__max_leaf_nodes": randint(7, 63),
         "classifier__learning_rate": loguniform(0.01, 0.25),
         "classifier__l2_regularization": loguniform(1e-4, 20),
         "classifier__min_samples_leaf": randint(10, 60)})

    if HAS_XGB:
        space["xgb"] = (make_model_pipeline(
            XGBClassifier(random_state=cfg.random_seed, n_jobs=1, eval_metric="auc",
                        tree_method="hist", scale_pos_weight=scale_pos_weight), cfg),
            {"classifier__n_estimators": randint(150, 700), "classifier__max_depth": randint(2, 7),
             "classifier__learning_rate": loguniform(0.01, 0.25), "classifier__subsample": [0.60, 0.75, 0.90, 1.0],
             "classifier__colsample_bytree": [0.50, 0.65, 0.80, 1.0], "classifier__min_child_weight": randint(1, 12),
             "classifier__reg_alpha": loguniform(1e-4, 10), "classifier__reg_lambda": loguniform(1e-3, 30)})
    return space


def tune_models(X_train, y_train, inner_cv, cfg: Config, groups_train=None) -> dict:
    tuned = {}
    for name, (estimator, params) in model_search_space(y_train, cfg).items():
        search = RandomizedSearchCV(estimator=estimator, param_distributions=params, n_iter=cfg.random_search_iter,
                                     scoring="roc_auc", cv=inner_cv, random_state=cfg.random_seed,
                                     n_jobs=cfg.n_jobs, refit=True, error_score=np.nan)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if groups_train is None:
                search.fit(X_train, y_train)
            else:
                search.fit(X_train, y_train, groups=groups_train)
        best_estimator = search.best_estimator_
        try:
            cv_kwargs = {"groups": groups_train} if groups_train is not None else {}
            oof_proba = cross_val_predict(clone(best_estimator), X_train, y_train, cv=inner_cv,
                                           method="predict_proba", n_jobs=cfg.n_jobs, **cv_kwargs)[:, 1]
            threshold = optimize_threshold(y_train, oof_proba, cfg.threshold_metric)
        except Exception:
            threshold = 0.5
        tuned[name] = {
            "estimator": best_estimator, "inner_cv_auc": float(search.best_score_),
            "best_params": search.best_params_, "threshold": threshold,
            "selected_features": pipeline_selected_features(best_estimator),
        }
    return tuned


def ensemble_member_items(tuned: dict, min_inner_auc: float) -> list:
    eligible = [(n, i) for n, i in tuned.items() if np.isfinite(i["inner_cv_auc"]) and i["inner_cv_auc"] >= min_inner_auc]
    if eligible:
        return eligible
    finite = [(n, i) for n, i in tuned.items() if np.isfinite(i["inner_cv_auc"])]
    return [max(finite, key=lambda item: item[1]["inner_cv_auc"])] if finite else list(tuned.items())


def ensemble_weights(member_items: list) -> np.ndarray:
    weights = np.array([max(info["inner_cv_auc"], 1e-6) for _, info in member_items], dtype=float)
    return weights / weights.sum()


def soft_vote_proba(tuned: dict, X: pd.DataFrame, min_inner_auc: float) -> np.ndarray:
    members = ensemble_member_items(tuned, min_inner_auc)
    weights = ensemble_weights(members)
    probs = np.array([info["estimator"].predict_proba(X)[:, 1] for _, info in members])
    return np.average(probs, axis=0, weights=weights)


def metric_row(y_true: np.ndarray, proba: np.ndarray, threshold: float) -> dict:
    # (patch v2) suppress sklearn's single-class UserWarning noise; behavior
    # is unchanged (roc_auc/pr_auc already resolve to NaN and get filtered
    # out downstream by confidence_interval's np.isfinite mask) -- this only
    # quiets the console spam when a validation fold happens to be single-class.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        roc_auc = roc_auc_score(y_true, proba) if len(np.unique(y_true)) == 2 else np.nan
        pr_auc = average_precision_score(y_true, proba) if (y_true == 1).any() else np.nan
        result = {"roc_auc": roc_auc, "pr_auc": pr_auc, "accuracy": accuracy_score(y_true, pred),
                  "f1": f1_score(y_true, pred, zero_division=0), "mcc": matthews_corrcoef(y_true, pred),
                  "sensitivity": sensitivity, "specificity": specificity,
                  "balanced_accuracy": balanced_accuracy_score(y_true, pred), "threshold": threshold,
                  "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}
    return result


In [ ]:
def make_outer_splits(y: np.ndarray, groups: np.ndarray, cfg: Config):

    use_groups = cfg.use_similarity_cv in ("always", "auto")
    if not use_groups:
        splitter = RepeatedStratifiedKFold(n_splits=cfg.outer_folds, n_repeats=cfg.outer_repeats,
                                            random_state=cfg.random_seed)
        for split_id, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y), start=1):
            repeat = (split_id - 1) // cfg.outer_folds + 1
            fold = (split_id - 1) % cfg.outer_folds + 1
            yield split_id, repeat, fold, train_idx, val_idx, "repeated_stratified_kfold"
        return

    n_unique_groups = len(np.unique(groups)) if groups is not None else 0
    effective_splits = min(cfg.outer_folds, n_unique_groups)
    if effective_splits < 2:
        raise ValueError(
            f"Cannot run group-aware CV: only {n_unique_groups} unique similarity "
            f"group(s) in this data but need >=2. Either this subset is too small/"
            f"too similarity-collapsed for group-aware CV, or set "
            f"use_similarity_cv='never' deliberately for this specific probe."
        )
    if effective_splits < cfg.outer_folds:
        print(f"[cv] WARNING: only {n_unique_groups} unique similarity groups available; "
              f"reducing outer_folds from {cfg.outer_folds} to {effective_splits} for this "
              f"run to avoid empty validation folds.")

    try:
        for repeat in range(1, cfg.outer_repeats + 1):
            splitter = StratifiedGroupKFold(n_splits=effective_splits, shuffle=True,
                                             random_state=cfg.random_seed + repeat - 1)
            for fold, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y, groups=groups), start=1):
                if len(val_idx) == 0 or len(train_idx) == 0:
                    print(f"[cv] Skipping degenerate fold (repeat {repeat}, fold {fold}) "
                          f"with an empty train/val partition.")
                    continue
                split_id = (repeat - 1) * effective_splits + fold
                yield split_id, repeat, fold, train_idx, val_idx, "stratified_group_kfold_similarity_aware"
    except ValueError:
        raise
    except Exception as exc:
        print(f"[cv] StratifiedGroupKFold unavailable/failed ({exc}); falling back to GroupKFold.")
        splitter = GroupKFold(n_splits=effective_splits)
        for fold, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y, groups=groups), start=1):
            if len(val_idx) == 0 or len(train_idx) == 0:
                continue
            yield fold, 1, fold, train_idx, val_idx, "group_kfold_similarity_aware"


def make_inner_cv(y_train: np.ndarray, groups_train: np.ndarray, cfg: Config, cv_mode: str):
    class_counts = np.bincount(y_train.astype(int), minlength=2)
    n_splits = min(cfg.inner_folds, int(class_counts.min()))
    if n_splits < 2:
        raise ValueError("Inner CV needs >=2 samples per class in the outer-training fold.")
    if "group" in cv_mode and groups_train is not None and len(np.unique(groups_train)) >= n_splits:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed)
        try:
            next(sgkf.split(np.zeros(len(y_train)), y_train, groups=groups_train))
            return sgkf, groups_train
        except Exception as exc:
            print(f"[cv] StratifiedGroupKFold failed for inner CV ({exc}); falling back to StratifiedKFold.")
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed), None


def run_nested_cv(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cfg: Config,
                   row_ids: np.ndarray = None) -> tuple:
    if row_ids is None:
        row_ids = np.arange(len(y))

    rows, full_records, oof_records = [], [], []
    for split_id, repeat, fold, train_idx, val_idx, cv_mode in make_outer_splits(y, groups, cfg):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        groups_train = groups[train_idx] if groups is not None else None
        inner_cv, inner_groups = make_inner_cv(y_train, groups_train, cfg, cv_mode)
        tuned = tune_models(X_train, y_train, inner_cv, cfg, groups_train=inner_groups)

        split_rows = []
        for model_name, info in tuned.items():
            proba = info["estimator"].predict_proba(X_val)[:, 1]
            metrics = metric_row(y_val, proba, info["threshold"])
            row = {"split": split_id, "repeat": repeat, "fold": fold, "cv_mode": cv_mode, "model": model_name,
                   "n_features_final": len(info["selected_features"]), "inner_cv_auc": info["inner_cv_auc"], **metrics}
            split_rows.append(row)
            full_records.append({"row": row, "selected_features": info["selected_features"],
                                  "best_params": info["best_params"]})
            for i, ridx in enumerate(val_idx):
                oof_records.append({"row_id": row_ids[ridx], "repeat": repeat, "model": model_name,
                                     "proba": float(proba[i]), "y_true": int(y_val[i])})

        members = ensemble_member_items(tuned, cfg.ensemble_min_inner_auc)
        ens_proba = soft_vote_proba(tuned, X_val, cfg.ensemble_min_inner_auc)
        ens_threshold = optimize_threshold(y_val, ens_proba, cfg.threshold_metric)
        ens_metrics = metric_row(y_val, ens_proba, ens_threshold)
        ens_selected = sorted({f for _, info in members for f in info["selected_features"]})
        ens_row = {"split": split_id, "repeat": repeat, "fold": fold, "cv_mode": cv_mode, "model": "weighted_ensemble",
                   "n_features_final": len(ens_selected), "inner_cv_auc": np.mean([i["inner_cv_auc"] for _, i in members]),
                   "ensemble_members": ",".join(n for n, _ in members), **ens_metrics}
        split_rows.append(ens_row)
        full_records.append({"row": ens_row, "selected_features": ens_selected,
                              "best_params": {"ensemble_members": [n for n, _ in members]}})
        for i, ridx in enumerate(val_idx):
            oof_records.append({"row_id": row_ids[ridx], "repeat": repeat, "model": "weighted_ensemble",
                                 "proba": float(ens_proba[i]), "y_true": int(y_val[i])})
        rows.extend(split_rows)

        auc_text = " | ".join(f"{r['model']}={r['roc_auc']:.3f}" for r in split_rows
                               if r["model"] in ("rf", "extra_trees", "xgb", "weighted_ensemble"))
        print(f"[repeat {repeat} fold {fold}] features={ens_row['n_features_final']} | {auc_text}")
    return pd.DataFrame(rows), full_records, pd.DataFrame(oof_records)


In [ ]:
def confidence_interval(values, confidence: float = 0.95) -> tuple:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    mean = float(np.mean(values))
    if len(values) == 1:
        return mean, np.nan, np.nan
    sem = np.std(values, ddof=1) / math.sqrt(len(values))
    margin = t.ppf((1 + confidence) / 2, df=len(values) - 1) * sem
    return mean, float(mean - margin), float(mean + margin)


def summarize_metrics(cv_df: pd.DataFrame, outdir: Path) -> pd.DataFrame:
    metric_cols = ["roc_auc", "pr_auc", "f1", "mcc", "sensitivity", "specificity", "balanced_accuracy"]
    rows = []
    for model_name, group in cv_df.groupby("model"):
        row = {"model": model_name, "n_splits": int(len(group))}
        for metric in metric_cols:
            mean, low, high = confidence_interval(group[metric].to_numpy())
            row[f"{metric}_mean"], row[f"{metric}_ci95_low"], row[f"{metric}_ci95_high"] = mean, low, high
            row[f"{metric}_std"] = float(group[metric].std())
        rows.append(row)
    summary = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
    summary.to_csv(outdir / "metric_summary_ci95.csv", index=False)
    return summary


def feature_selection_stability(full_records: list, outdir: Path) -> pd.DataFrame:
    main_records = [r for r in full_records if r["row"]["model"] != "weighted_ensemble"]
    all_feats = sorted({f for r in main_records for f in r["selected_features"]})
    counts = pd.Series(0, index=all_feats, dtype=int)
    for record in main_records:
        counts.loc[record["selected_features"]] += 1
    denom = max(len(main_records), 1)
    stability = (counts / denom).sort_values(ascending=False).rename("selection_frequency").to_frame()
    stability["family"] = [feature_family(f) for f in stability.index]
    stability.index.name = "feature"
    stability.to_csv(outdir / "feature_selection_stability.csv")
    return stability


def subgroup_performance_report(df: pd.DataFrame, oof_df: pd.DataFrame,
                                 subgroup_cols=("MHC_Class_Used", "Organism"),
                                 model_name: str = "weighted_ensemble",
                                 min_group_size: int = 10) -> pd.DataFrame:

    sub = oof_df[oof_df["model"] == model_name].copy()
    agg = sub.groupby("row_id").agg(proba=("proba", "mean"), y_true=("y_true", "first"),
                                     n_folds=("proba", "size")).reset_index()
    meta = df.reset_index(drop=True).reset_index().rename(columns={"index": "row_id"})
    keep_cols = ["row_id"] + [c for c in subgroup_cols if c in meta.columns]
    merged = agg.merge(meta[keep_cols], on="row_id", how="left")

    rows = []
    for col in subgroup_cols:
        if col not in merged.columns:
            continue
        for val, g in merged.groupby(col):
            if g["y_true"].nunique() < 2 or len(g) < min_group_size:
                continue
            rows.append({"subgroup_column": col, "subgroup_value": val, "n": int(len(g)),
                         "n_positive": int((g["y_true"] == 1).sum()),
                         "n_negative": int((g["y_true"] == 0).sum()),
                         "roc_auc": float(roc_auc_score(g["y_true"], g["proba"]))})
    out = pd.DataFrame(rows).sort_values(["subgroup_column", "roc_auc"], ascending=[True, False])
    return out


def quick_leakage_ablation(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cfg: Config) -> dict:
    probe_kwargs = {**asdict(cfg), "random_search_iter": 10, "outer_repeats": 1}

    print("[leakage ablation] Running HONEST (similarity-group-aware) CV probe...")
    honest_cfg = Config(**{**probe_kwargs, "use_similarity_cv": "always"})
    honest_df, _, _ = run_nested_cv(X, y, groups, honest_cfg)
    honest_auc = float(honest_df.loc[honest_df["model"] == "weighted_ensemble", "roc_auc"].mean())

    print("[leakage ablation] Running NAIVE (non-grouped) CV probe -- for reporting the gap ONLY...")
    naive_cfg = Config(**{**probe_kwargs, "use_similarity_cv": "never"})
    naive_df, _, _ = run_nested_cv(X, y, groups, naive_cfg)
    naive_auc = float(naive_df.loc[naive_df["model"] == "weighted_ensemble", "roc_auc"].mean())

    result = {
        "honest_group_aware_auc_probe": honest_auc,
        "naive_non_grouped_auc_probe": naive_auc,
        "estimated_leakage_inflation": naive_auc - honest_auc,
        "note": "These are LIGHT-BUDGET probes (random_search_iter=10, outer_repeats=1) for "
                "quantifying leakage only. Report cv_fold_results.csv / metric_summary_ci95.csv "
                "(full-budget, similarity-group-aware) as the model's actual performance.",
    }
    print(f"[leakage ablation] {result}")
    return result


In [ ]:
def train_final_model(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cfg: Config):
    """Tune once more on ALL resolved data to produce the deployable artifact.
    The honest performance estimate is cv_fold_results.csv, NOT this refit."""
    class_counts = np.bincount(y.astype(int), minlength=2)
    n_splits = min(cfg.inner_folds, int(class_counts.min()))
    if n_splits < 2:
        raise ValueError(f"Final tuning needs >=2 samples per class; found {int(class_counts.min())} in minority class.")

    if len(np.unique(groups)) >= n_splits:
        cv, cv_groups = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed), groups
    else:
        cv, cv_groups = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed), None

    tuned = tune_models(X, y, cv, cfg, groups_train=cv_groups)
    members = ensemble_member_items(tuned, cfg.ensemble_min_inner_auc)
    weights = ensemble_weights(members)
    estimators = {name: info["estimator"] for name, info in members}
    threshold = float(np.mean([info["threshold"] for _, info in members]))
    final_model = WeightedSoftVotingEnsemble(estimators=estimators, weights=weights, threshold=threshold)
    return final_model, tuned


def predict_new_peptides(new_csv: str, model_pkl: str, output_csv: str = None) -> pd.DataFrame:
    """Load a final_model.pkl artifact and score new peptides (future validation)."""
    with open(model_pkl, "rb") as f:
        artifact = pickle.load(f)
    model = artifact["model"]
    feature_columns = artifact["feature_columns"]
    cfg = Config(**artifact["config"])

    df = pd.read_csv(new_csv)
    if cfg.label_col not in df.columns:
        df[cfg.label_col] = 0  # placeholder only; not used for scoring

    feat_df = build_feature_matrix(df, cfg)
    drop_cols = [c for c in [cfg.id_col, cfg.label_col, cfg.seq_col] if c in feat_df.columns]
    X = feat_df.drop(columns=drop_cols)
    for col in feature_columns:
        if col not in X.columns:
            X[col] = 0.0
    X = X[feature_columns]

    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= model.threshold).astype(int)
    out = feat_df[[c for c in [cfg.id_col, cfg.seq_col] if c in feat_df.columns]].copy()
    out["predicted_probability"] = proba
    out["predicted_label"] = pred
    if output_csv:
        out.to_csv(output_csv, index=False)
        print(f"[predict] saved {len(out)} predictions to {output_csv}")
    return out


In [ ]:
def compute_shap_explanations(X: pd.DataFrame, y: np.ndarray, cfg: Config, outdir: Path):

    if not HAS_SHAP:
        print("[shap] shap not installed (`pip install shap`); skipping.")
        return None

    if HAS_XGB:
        base_estimator = XGBClassifier(random_state=cfg.random_seed, n_jobs=cfg.n_jobs, eval_metric="auc",
                                        tree_method="hist", n_estimators=400, max_depth=4,
                                        learning_rate=0.05, subsample=0.8, colsample_bytree=0.7)
        model_tag = "xgb"
    else:
        base_estimator = RandomForestClassifier(random_state=cfg.random_seed, n_jobs=cfg.n_jobs,
                                                  class_weight="balanced_subsample",
                                                  n_estimators=600, max_depth=10)
        model_tag = "rf"

    pipe = make_model_pipeline(base_estimator, cfg)
    pipe.fit(X, y)
    selected = pipeline_selected_features(pipe)

    X_sel = pipe.named_steps["var_corr"].transform(X)
    X_sel = pipe.named_steps["mi"].transform(X_sel)
    X_scaled = pipe.named_steps["scaler"].transform(X_sel)
    X_scaled_df = pd.DataFrame(X_scaled, columns=selected, index=X.index)

    n_bg = min(cfg.shap_max_background, len(X_scaled_df))
    rng = np.random.RandomState(cfg.random_seed)
    sample_idx = rng.choice(len(X_scaled_df), n_bg, replace=False)
    X_sample = X_scaled_df.iloc[sample_idx]

    explainer = shap.TreeExplainer(pipe.named_steps["classifier"])
    shap_values = explainer.shap_values(X_sample)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    shap_df = pd.DataFrame(shap_values, columns=selected)
    shap_df.to_csv(outdir / "shap_values.csv", index=False)
    mean_abs = shap_df.abs().mean().sort_values(ascending=False)
    mean_abs.to_csv(outdir / "shap_feature_importance.csv", header=["mean_abs_shap"])

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
        plt.tight_layout()
        plt.savefig(outdir / "shap_summary_plot.png", dpi=200, bbox_inches="tight")
        plt.close()
        print(f"[shap] saved shap_summary_plot.png")
    except Exception as e:
        print(f"[shap] plot generation failed (non-fatal): {e}")

    with open(outdir / "shap_explainer_model.pkl", "wb") as f:
        pickle.dump({"pipe": pipe, "explainer_model_tag": model_tag,
                     "selected_features": selected, "config": asdict(cfg)}, f)

    print(f"[shap] explainer model: {model_tag} | top 10 features by mean |SHAP|:")
    print(mean_abs.head(10).to_string())
    return {"explainer_model": model_tag, "top_features": mean_abs.head(20).to_dict()}


In [ ]:
def compute_lime_explanations(X: pd.DataFrame, y: np.ndarray, cfg: Config, outdir: Path,
                               shap_model_path: Path = None):
    if not HAS_LIME:
        print("[lime] lime not installed (`pip install lime`); skipping.")
        return None

    if shap_model_path is not None and Path(shap_model_path).exists():
        with open(shap_model_path, "rb") as f:
            bundle = pickle.load(f)
        pipe, selected = bundle["pipe"], bundle["selected_features"]
    else:
        base_estimator = (XGBClassifier(random_state=cfg.random_seed, n_jobs=cfg.n_jobs, eval_metric="auc",
                                         tree_method="hist", n_estimators=400, max_depth=4,
                                         learning_rate=0.05, subsample=0.8, colsample_bytree=0.7)
                           if HAS_XGB else
                           RandomForestClassifier(random_state=cfg.random_seed, n_jobs=cfg.n_jobs,
                                                   class_weight="balanced_subsample",
                                                   n_estimators=600, max_depth=10))
        pipe = make_model_pipeline(base_estimator, cfg)
        pipe.fit(X, y)
        selected = pipeline_selected_features(pipe)

    X_sel_all = pipe.named_steps["var_corr"].transform(X)
    X_sel_all = pipe.named_steps["mi"].transform(X_sel_all)
    X_scaled_all = pipe.named_steps["scaler"].transform(X_sel_all)
    X_scaled_all = np.asarray(X_scaled_all)

    classifier = pipe.named_steps["classifier"]
    predict_fn = lambda arr: classifier.predict_proba(arr)

    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_scaled_all, feature_names=selected,
        class_names=["non_immunogenic", "immunogenic"], mode="classification",
        random_state=cfg.random_seed)

    proba = pipe.predict_proba(X)[:, 1]
    rng = np.random.RandomState(cfg.random_seed)
    tp_idx = np.where((y == 1) & (proba > 0.65))[0]
    tn_idx = np.where((y == 0) & (proba < 0.35))[0]
    borderline_idx = np.where(np.abs(proba - 0.5) < 0.08)[0]

    picks = []
    for pool, tag in [(tp_idx, "confident_positive"), (tn_idx, "confident_negative"),
                       (borderline_idx, "borderline")]:
        if len(pool) > 0:
            picks.append((int(rng.choice(pool)), tag))

    results = []
    for idx, tag in picks[:cfg.lime_n_explanations]:
        exp = explainer.explain_instance(X_scaled_all[idx], predict_fn, num_features=10)
        html_path = outdir / f"lime_{tag}_row{idx}.html"
        try:
            exp.save_to_file(str(html_path))
        except Exception as e:
            print(f"[lime] could not save html for row {idx} (non-fatal): {e}")
        results.append({"row_index": idx, "tag": tag, "true_label": int(y[idx]),
                         "predicted_proba": float(proba[idx]), "explanation": exp.as_list()})

    with open(outdir / "lime_explanations.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"[lime] saved {len(results)} local explanations to lime_explanations.json")
    return results


In [ ]:
def run_full_pipeline(cfg: Config):
    outdir = Path(cfg.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    (outdir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

    # 1. Load data
    df = pd.read_csv(cfg.input_csv)
    print(f"[load] {len(df)} rows from {cfg.input_csv}")

    # 2. Restrict to MHC assay-supported peptides
    df, restrict_report = restrict_to_assay_supported(df, cfg)

    # 2b. NEW: restrict to direct T-cell-assay-confirmed peptides
    df, tcell_report = restrict_to_tcell_assay(df, cfg)

    # 3. Similarity clustering (on the FINAL filtered set)
    seqs = df[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    groups = compute_similarity_groups(seqs, cfg.similarity_threshold)

    # 4. Resolve label conflicts
    if cfg.resolve_conflicts:
        df, groups, conflict_report = resolve_label_conflicts(df, groups, cfg)
    else:
        conflict_report = {"resolve_conflicts": False}

    # 5. Drop invalid sequences, keep groups aligned
    seqs_clean = df[cfg.seq_col].astype(str).map(clean_sequence)
    valid_mask = seqs_clean.map(is_valid_peptide)
    if (~valid_mask).any():
        n_inv = int((~valid_mask).sum())
        print(f"[pipeline] Dropping {n_inv} invalid sequences before feature extraction.")
        df = df.loc[valid_mask].reset_index(drop=True)
        groups = groups[valid_mask.to_numpy()]

    row_ids = np.arange(len(df))  # stable ids for OOF/subgroup reporting

    # 6. Build feature matrix (ESM-2 cached automatically)
    feat_df = build_feature_matrix(df, cfg)
    if feat_df.empty:
        raise ValueError("No valid peptides remaining after filtering.")

    drop_cols = [c for c in [cfg.id_col, cfg.label_col, cfg.seq_col] if c in feat_df.columns]
    X = feat_df.drop(columns=drop_cols)
    y = feat_df[cfg.label_col].to_numpy().astype(int)
    groups = groups[:len(feat_df)]
    row_ids = row_ids[:len(feat_df)]

    # 7. Nested, similarity-aware CV  (the HONEST performance estimate)
    cv_df, full_records, oof_df = run_nested_cv(X, y, groups, cfg, row_ids=row_ids)
    cv_df.to_csv(outdir / "cv_fold_results.csv", index=False)
    oof_df.to_csv(outdir / "oof_predictions.csv", index=False)
    summary = summarize_metrics(cv_df, outdir)
    feature_selection_stability(full_records, outdir)

    # 7b. NEW: honest subgroup breakdown (MHC class, organism if present)
    subgroup_cols = [c for c in ["MHC_Class_Used", "Organism"] if c in df.columns]
    subgroup_df = pd.DataFrame()
    if subgroup_cols:
        subgroup_df = subgroup_performance_report(df, oof_df, subgroup_cols=subgroup_cols)
        subgroup_df.to_csv(outdir / "subgroup_performance.csv", index=False)
        print("\n[subgroup performance]\n" + subgroup_df.to_string(index=False))

    # 8. Final deployable model (fit on all resolved data; NOT the perf estimate)
    final_model, final_tuned = train_final_model(X, y, groups, cfg)
    with open(outdir / "final_model.pkl", "wb") as f:
        pickle.dump({
            "model": final_model,
            "feature_columns": X.columns.tolist(),
            "config": asdict(cfg),
            "member_inner_cv_auc": {n: i["inner_cv_auc"] for n, i in final_tuned.items()},
        }, f)
    print(f"[save] final_model.pkl written to {outdir}")

    # 9. NEW: explainability artifacts for the paper (SHAP + LIME)
    shap_result, lime_result = None, None
    if cfg.save_explainability_artifacts:
        shap_result = compute_shap_explanations(X, y, cfg, outdir)
        lime_result = compute_lime_explanations(
            X, y, cfg, outdir, shap_model_path=outdir / "shap_explainer_model.pkl")

    # 10. Report
    ens_auc = None
    if "weighted_ensemble" in summary["model"].values:
        ens_auc = float(summary.loc[summary["model"] == "weighted_ensemble", "roc_auc_mean"].iloc[0])
    report = {
        "restrict_to_assay_support": restrict_report,
        "restrict_to_tcell_assay": tcell_report,
        "conflict_resolution": conflict_report,
        "n_features": int(X.shape[1]),
        "n_peptides_final": int(len(feat_df)),
        "ensemble_roc_auc_mean": ens_auc,
        "shap_computed": shap_result is not None,
        "lime_computed": lime_result is not None,
    }
    (outdir / "run_report.json").write_text(json.dumps(report, indent=2))

    print(f"\n[done] {report}")
    print("\n" + "=" * 70)
    print(summary.to_string(index=False))
    print("=" * 70)

    return {
        "df": df, "X": X, "y": y, "groups": groups, "cv_df": cv_df,
        "oof_df": oof_df, "summary": summary, "subgroup_df": subgroup_df,
        "final_model": final_model, "shap_result": shap_result, "lime_result": lime_result,
        "report": report,
    }


In [ ]:
def run_leakage_ablation_report(cfg: Config, outdir: Path):

    df = pd.read_csv(cfg.input_csv)
    df, _ = restrict_to_assay_supported(df, cfg)
    df, _ = restrict_to_tcell_assay(df, cfg)
    seqs = df[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    groups = compute_similarity_groups(seqs, cfg.similarity_threshold)
    if cfg.resolve_conflicts:
        df, groups, _ = resolve_label_conflicts(df, groups, cfg)
    valid_mask = df[cfg.seq_col].astype(str).map(clean_sequence).map(is_valid_peptide)
    df = df.loc[valid_mask].reset_index(drop=True)
    groups = groups[valid_mask.to_numpy()][:len(df)]

    feat_df = build_feature_matrix(df, cfg)
    drop_cols = [c for c in [cfg.id_col, cfg.label_col, cfg.seq_col] if c in feat_df.columns]
    X = feat_df.drop(columns=drop_cols)
    y = feat_df[cfg.label_col].to_numpy().astype(int)
    groups = groups[:len(feat_df)]

    result = quick_leakage_ablation(X, y, groups, cfg)
    (Path(outdir) / "leakage_ablation.json").write_text(json.dumps(result, indent=2))
    return result


In [ ]:
def maybe_colab_upload(default_filename: str):

    if Path(default_filename).exists():
        return
    try:
        from google.colab import files  # type: ignore
        print(f"[colab] '{default_filename}' not found -- please upload it:")
        files.upload()
    except ImportError:
        pass  # not running in Colab; assume the file is already on disk


def zip_and_offer_download(outdir: str):
    """Optional convenience: zip the report folder and trigger a Colab
    download. No-op outside Colab."""
    import shutil
    zip_path = shutil.make_archive(outdir, "zip", root_dir=outdir)
    print(f"[package] wrote {zip_path}")
    try:
        from google.colab import files  # type: ignore
        files.download(zip_path)
    except ImportError:
        print(f"[package] (not in Colab) find your results at: {zip_path}")


## Step 5: Diagnostics (no training needed)

Four tables:
1. **Cluster and cohort table:** when the threshold changes, how many clusters, how many singletons, how many conflicting groups, and how large the cohort is.
2. **Residual neighbours:** for each peptide in the baseline cohort, how identical its closest neighbour **in another 0.80-cluster** is, and how many peptides share a k-aa stretch with a peptide of another cluster. (This measures the "gap" of the 80% cutoff.)
3. **By length:** how many edits the same identity cutoff equals at each length.
4. **External cutoff sweep:** the baseline model's AUC when the cutoff for removing external peptides near the training set changes.

**Sanity check:** the `0.80` row must give a cohort of **718**.

In [ ]:
import numpy as np, pandas as pd
from dataclasses import replace
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.metrics import roc_auc_score, average_precision_score


def make_cfg(outdir, use_sim_cv="auto", **over):
    return Config(input_csv=MAIN_CSV, outdir=str(outdir), n_jobs=N_JOBS,
                  use_esm_embeddings=USE_ESM, esm_model_name=ESM_MODEL,
                  random_search_iter=SEARCH_ITER, outer_repeats=OUTER_REPEATS,
                  outer_folds=OUTER_FOLDS, inner_folds=INNER_FOLDS,
                  save_explainability_artifacts=False, require_tcell_assay=True,
                  use_similarity_cv=use_sim_cv, esm_batch_size=ESM_BATCH_SIZE, **over)


def edit_sim(a, b=None):
    return process.cdist(a, a if b is None else b, scorer=Levenshtein.normalized_similarity, workers=-1)


def components(adj):
    return connected_components(csr_matrix(adj), directed=False)[1]


def kmer_share_matrix(peps, k):
    inv = {}
    for i, p in enumerate(peps):
        for m in {p[j:j + k] for j in range(len(p) - k + 1)}:
            inv.setdefault(m, []).append(i)
    A = np.zeros((len(peps), len(peps)), dtype=bool)
    for lst in inv.values():
        if len(lst) > 1:
            idx = np.array(lst); A[np.ix_(idx, idx)] = True
    np.fill_diagonal(A, False)
    return A


def resolve_cohort(cfg):
    """Pipeline steps 1-4: load, restrict, cluster at cfg.similarity_threshold, resolve conflicts."""
    d0 = pd.read_csv(cfg.input_csv)
    d0, _ = restrict_to_assay_supported(d0, cfg)
    d0, _ = restrict_to_tcell_assay(d0, cfg)
    seqs = d0[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    g = compute_similarity_groups(seqs, cfg.similarity_threshold)
    d1, g1, rep = resolve_label_conflicts(d0, g, cfg)
    return d1.reset_index(drop=True), np.asarray(g1), rep, d0


cfg_b = make_cfg(OUT / "cohort")
coh, g80, rep80, raw = resolve_cohort(cfg_b)
assert len(coh) == 718, f"baseline cohort {len(coh)} != 718"

# ---- 1. cluster / cohort table over thresholds ----
raw_seqs = raw[cfg_b.seq_col].astype(str).map(clean_sequence).tolist()
S_raw = edit_sim(raw_seqs)
rows = []
for thr in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00]:
    g = components(S_raw >= thr - 1e-12); sizes = np.bincount(g)
    d2, g2, rep = resolve_label_conflicts(raw.copy(), g, cfg_b)
    y2 = d2[cfg_b.label_col].astype(int)
    rows.append({"threshold": thr, "clusters": len(sizes), "singleton_pct": round(100 * (sizes == 1).mean(), 1),
                 "largest_cluster": int(sizes.max()),
                 "conflicting_groups": rep["n_conflicting_groups_relabeled"] + rep["n_conflicting_groups_dropped"],
                 "peptides_dropped": rep["n_peptides_dropped_as_contradictory"], "cohort_n": len(d2),
                 "n_pos": int(y2.sum()), "n_classI": int((d2["MHC_Class_Used"] == "I").sum())})
diag_cluster = pd.DataFrame(rows); diag_cluster.to_csv(OUT / "diag_cluster_table.csv", index=False)
display(diag_cluster) if "display" in globals() else print(diag_cluster.to_string(index=False))

In [ ]:
# ---- 2. residual neighbours on the baseline cohort (718), 0.80 clusters ----
P = coh[cfg_b.seq_col].astype(str).map(clean_sequence).tolist()
S_coh = edit_sim(P)
cross = np.where(g80[:, None] == g80[None, :], 0.0, S_coh)
mx = cross.max(axis=1)
res_rows = [{"measure": f"edit identity to a peptide in another 0.80-cluster >= {thr}",
             "n_peptides": int((mx >= thr).sum()), "pct": round(100 * (mx >= thr).mean(), 1)}
            for thr in (0.60, 0.70, 0.75)]
isII = (coh["MHC_Class_Used"] == "II").to_numpy()
for k in (9, 8, 7):
    Kk = kmer_share_matrix(P, k)
    diff = Kk & (g80[:, None] != g80[None, :])
    has = diff.any(axis=1)
    res_rows.append({"measure": f"shares a {k}-aa stretch with a peptide of another 0.80-cluster",
                     "n_peptides": int(has.sum()), "pct": round(100 * has.mean(), 1),
                     "of_which_classII": int((has & isII).sum())})
diag_resid = pd.DataFrame(res_rows); diag_resid.to_csv(OUT / "diag_residual_neighbours.csv", index=False)
print(diag_resid.to_string(index=False))

# ---- 3. what an identity cutoff means in edits, by length ----
len_rows = [{"length": L, **{f"max_edits_at_{thr}": int(math.floor((1 - thr) * L + 1e-9)) for thr in (0.5, 0.6, 0.7, 0.8, 0.9)}}
            for L in (9, 12, 15, 20, 25)]
diag_len = pd.DataFrame(len_rows); diag_len.to_csv(OUT / "diag_edits_by_length.csv", index=False)
print("\n", diag_len.to_string(index=False))

In [ ]:
# ---- 4. external cutoff sweep (needs the baseline external prediction file) ----
def boot_auc(y, p, n_boot=N_BOOT, seed=42):
    y, p = np.asarray(y), np.asarray(p); rng = np.random.RandomState(seed); vals = []
    for _ in range(n_boot):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        vals.append(roc_auc_score(y[i], p[i]))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return roc_auc_score(y, p), lo, hi

if Path(BASE_EXT_PRED).exists():
    E0 = pd.read_csv(BASE_EXT_PRED)
    sweep = []
    for cut in [None, 0.90, 0.80, 0.70, 0.60, 0.50]:
        k = E0 if cut is None else E0[E0["max_id_to_train"] < cut]
        a, lo, hi = boot_auc(k["label"], k["predicted_probability"])
        kc = k[k["MHC_Class_Used"] == "I"]
        ac, loc, hic = boot_auc(kc["label"], kc["predicted_probability"])
        sweep.append({"removal_cutoff": "none" if cut is None else cut, "n": len(k), "removed": len(E0) - len(k),
                      "AUC": a, "AUC_lo": lo, "AUC_hi": hi, "classI_n": len(kc), "classI_AUC": ac,
                      "classI_lo": loc, "classI_hi": hic})
    ext_sweep = pd.DataFrame(sweep); ext_sweep.to_csv(OUT / "diag_external_cutoff_sweep.csv", index=False)
    print(ext_sweep.round(3).to_string(index=False))
else:
    ext_sweep = None
    print("BASE_EXT_PRED not found, sweep skipped. The baseline will be re-run in Design B.")

## Step 6: Design A, CV-only (cohort fixed at 718)

In [ ]:
# ---- cohort features built once, reused by every Design A scenario ----
def prepare_cohort_features(cfg):
    d1, g1, rep, _ = resolve_cohort(cfg)
    seqs_clean = d1[cfg.seq_col].astype(str).map(clean_sequence)
    valid = seqs_clean.map(is_valid_peptide)
    if (~valid).any():
        d1 = d1.loc[valid].reset_index(drop=True); g1 = g1[valid.to_numpy()]
    feat = build_feature_matrix(d1, cfg)
    assert len(feat) == len(d1), "feature rows != cohort rows"
    drop_cols = [c for c in [cfg.id_col, cfg.label_col, cfg.seq_col] if c in feat.columns]
    return d1, feat.drop(columns=drop_cols), feat[cfg.label_col].to_numpy().astype(int), g1

A_df, A_X, A_y, A_g80 = prepare_cohort_features(cfg_b)
A_P = A_df[cfg_b.seq_col].astype(str).map(clean_sequence).tolist()
A_S = edit_sim(A_P)
A_K = {k: kmer_share_matrix(A_P, k) for k in (9, 8)}
print("cohort", len(A_y), "| positives", int(A_y.sum()), "| features", A_X.shape[1])

A_GROUPS = {"cv_0.80": (A_g80, "auto"), "naive": (A_g80, "never")}
for thr in (0.90, 0.70, 0.60, 0.50):
    A_GROUPS[f"cv_{thr:.2f}"] = (components(A_S >= thr - 1e-12), "auto")
for k in (9, 8):
    A_GROUPS[f"core{k}"] = (components((A_S >= 0.80 - 1e-12) | A_K[k]), "auto")
for nm, (g_, _) in A_GROUPS.items():
    sz = np.bincount(g_); print(f"{nm:9s} clusters={len(sz):4d} singleton%={100*(sz==1).mean():5.1f} largest={sz.max()}")

In [ ]:
def exposure(groups, mode, cfg_s):
    """Fraction of validation peptides that have a training-fold neighbour (edit>=0.70, >=0.75, shared 9-mer)."""
    n_tot = 0; acc = {"exp_edit70": 0.0, "exp_edit75": 0.0, "exp_share9": 0.0, "exp_share8": 0.0}
    for split_id, repeat, fold, tr, va, cv_mode in make_outer_splits(A_y, groups, cfg_s):
        sub = A_S[np.ix_(va, tr)]
        acc["exp_edit70"] += float((sub.max(axis=1) >= 0.70).sum())
        acc["exp_edit75"] += float((sub.max(axis=1) >= 0.75).sum())
        acc["exp_share9"] += float(A_K[9][np.ix_(va, tr)].any(axis=1).sum())
        acc["exp_share8"] += float(A_K[8][np.ix_(va, tr)].any(axis=1).sum())
        n_tot += len(va)
    return {k: v / n_tot for k, v in acc.items()}


def run_cv_scenario(name):
    groups, mode = A_GROUPS[name]
    outdir = OUT / "A" / name; outdir.mkdir(parents=True, exist_ok=True)
    rowf = outdir / "row.json"
    if rowf.exists():
        print(f"[skip] {name}"); return json.load(open(rowf))
    t0 = time.time()
    cfg_s = make_cfg(outdir, use_sim_cv=mode)
    cv_df, _, oof_df = run_nested_cv(A_X, A_y, groups, cfg_s, row_ids=np.arange(len(A_y)))
    cv_df.to_csv(outdir / "cv_fold_results.csv", index=False)
    oof_df.to_csv(outdir / "oof_predictions.csv", index=False)
    s = summarize_metrics(cv_df, outdir).set_index("model")
    sz = np.bincount(groups)
    row = {"scenario": name, "n": len(A_y), "n_clusters": int(len(sz)), "largest_cluster": int(sz.max())}
    for m, tag in (("weighted_ensemble", "ens"), ("extra_trees", "et")):
        for k_ in ("roc_auc", "pr_auc", "mcc"):
            row[f"{tag}_{k_}"] = float(s.loc[m, f"{k_}_mean"])
        row[f"{tag}_auc_lo"] = float(s.loc[m, "roc_auc_ci95_low"]); row[f"{tag}_auc_hi"] = float(s.loc[m, "roc_auc_ci95_high"])
    row.update(exposure(groups, mode, cfg_s))
    # pooled OOF AUC by class (ensemble)
    oe = oof_df[oof_df["model"] == "weighted_ensemble"].groupby("row_id")["proba"].mean().sort_index()
    cls = A_df["MHC_Class_Used"].to_numpy()
    for c_, tag in (("I", "classI"), ("II", "classII")):
        m_ = cls == c_
        row[f"ens_pooled_{tag}_auc"] = float(roc_auc_score(A_y[m_], oe.to_numpy()[m_]))
    row["ens_pooled_auc"] = float(roc_auc_score(A_y, oe.to_numpy()))
    row["minutes"] = round((time.time() - t0) / 60, 1)
    json.dump(row, open(rowf, "w"), indent=2)
    return row

A_res = {}
if RUN_DESIGN_A:
    for nm in A_RUN:
        print("\n" + "=" * 70 + f"\nDESIGN A: {nm}\n" + "=" * 70)
        try:
            A_res[nm] = run_cv_scenario(nm)
        except Exception:
            traceback.print_exc(); print(f"!!! {nm} failed")
    if "cv_0.80" in A_res:
        d_ = A_res["cv_0.80"]["ens_roc_auc"] - PREV_BASE_AUC
        print(f"\nReproduction check: cv_0.80 ens AUC = {A_res['cv_0.80']['ens_roc_auc']:.4f} vs earlier {PREV_BASE_AUC:.4f} (difference {d_:+.4f})",
              "OK" if abs(d_) < 0.005 else "<<< does not match, something has changed")

### Design A: result table

- `d_ens_auc`: difference in ensemble AUC from `cv_0.80` (fold mean).
- `d_pooled_lo/hi`: paired bootstrap 95% CI of the pooled OOF AUC on **the same peptides** (possible because the cohort is fixed).
- `robust`: `|d_ens_auc| ≤ TOL`. `naive` is a reference only.
- `exp_*`: the fraction of validation peptides that have a close neighbour in the training fold.

In [ ]:
def pooled_oof(name, model):
    o = pd.read_csv(OUT / "A" / name / "oof_predictions.csv")
    return o[o["model"] == model].groupby("row_id")["proba"].mean().sort_index().to_numpy()

A_tab = pd.DataFrame(list(A_res.values()))
if "cv_0.80" in A_res and len(A_tab):
    base_row = A_res["cv_0.80"]; rng = np.random.RandomState(11)
    for model, tag in (("weighted_ensemble", "ens"), ("extra_trees", "et")):
        pb = pooled_oof("cv_0.80", model); dd = []
        for nm in A_tab["scenario"]:
            if nm == "cv_0.80": dd.append((0.0, np.nan, np.nan)); continue
            ps = pooled_oof(nm, model); dif = []
            for _ in range(N_BOOT):
                i = rng.randint(0, len(A_y), len(A_y))
                if len(np.unique(A_y[i])) < 2: continue
                dif.append(roc_auc_score(A_y[i], ps[i]) - roc_auc_score(A_y[i], pb[i]))
            dd.append((roc_auc_score(A_y, ps) - roc_auc_score(A_y, pb), *np.percentile(dif, [2.5, 97.5])))
        A_tab[[f"d_pooled_{tag}", f"d_pooled_{tag}_lo", f"d_pooled_{tag}_hi"]] = pd.DataFrame(dd, index=A_tab.index)
    A_tab["d_ens_auc"] = A_tab["ens_roc_auc"] - base_row["ens_roc_auc"]
    A_tab["robust"] = [("reference" if nm == "naive" else bool(abs(d) <= TOL)) for nm, d in zip(A_tab["scenario"], A_tab["d_ens_auc"])]
A_tab.to_csv(OUT / "designA_table.csv", index=False)
cols = ["scenario", "n_clusters", "largest_cluster", "ens_roc_auc", "ens_auc_lo", "ens_auc_hi", "et_roc_auc",
        "ens_pooled_classI_auc", "ens_pooled_classII_auc", "exp_edit70", "exp_edit75", "exp_share9",
        "d_ens_auc", "d_pooled_ens", "d_pooled_ens_lo", "d_pooled_ens_hi", "d_pooled_et", "d_pooled_et_lo", "d_pooled_et_hi", "robust"]
A_tab[[c for c in cols if c in A_tab.columns]].round(3)

In [ ]:
import matplotlib.pyplot as plt
order = [s for s in ["naive", "cv_0.90", "cv_0.80", "cv_0.70", "cv_0.60", "cv_0.50", "core9", "core8"] if s in A_tab["scenario"].values]
T = A_tab.set_index("scenario").loc[order]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(T))
for col, lo, hi, lab, off in (("ens_roc_auc", "ens_auc_lo", "ens_auc_hi", "Weighted ensemble", -0.08), ("et_roc_auc", "et_auc_lo", "et_auc_hi", "Extra Trees", 0.08)):
    y_ = T[col].to_numpy(float); err = np.vstack([y_ - T[lo].to_numpy(float), T[hi].to_numpy(float) - y_]) if lo in T else None
    ax[0].errorbar(x + off, y_, yerr=err, fmt="o", capsize=3, label=lab)
ax[0].set_xticks(x); ax[0].set_xticklabels(order, rotation=30, ha="right"); ax[0].set_ylabel("Group-aware CV ROC-AUC")
ax[0].set_title("Internal AUC vs. CV grouping"); ax[0].legend()
for col, lab in (("exp_edit70", "edit identity ≥ 0.70"), ("exp_edit75", "edit identity ≥ 0.75"), ("exp_share9", "shares a 9-aa stretch")):
    ax[1].plot(x, T[col].to_numpy(float), "o-", label=lab)
ax[1].set_xticks(x); ax[1].set_xticklabels(order, rotation=30, ha="right"); ax[1].set_ylabel("Fraction of validation peptides")
ax[1].set_title("Residual neighbours in the training fold"); ax[1].legend()
plt.tight_layout(); plt.savefig(OUT / "figR24_designA.png", dpi=200); plt.show()

## Step 7: Design B, full pipeline (the cohort changes too)

`similarity_threshold` changes both the CV grouping and the conflict clusters. A final model is built and scores **the same external cohort**. External peptides that are ≥ that scenario's threshold identical to its training set are removed.

If `BASE_EXT_PRED` does not exist, the baseline (`sim_0.80`) is also run here (this adds one full scenario of time).

In [ ]:
EXT = pd.read_csv(EXT_CSV)
EXT["Peptide"] = EXT["Peptide"].astype(str).str.strip().str.upper()
EXT = EXT[EXT["Assay"].astype(str).str.contains("T cell assay", case=False, na=False)]
EXT = EXT[EXT["Peptide"].map(is_valid_peptide)].drop_duplicates("Peptide").reset_index(drop=True)
print("external peptides:", len(EXT), "(expected 3572)")

def run_B(name, thr):
    outdir = OUT / "B" / name; rowf = outdir / "row.json"
    if rowf.exists():
        print(f"[skip] {name}"); return json.load(open(rowf))
    t0 = time.time()
    cfg_s = make_cfg(outdir, similarity_threshold=thr)
    res = run_full_pipeline(cfg_s)
    s = res["summary"].set_index("model")
    row = {"scenario": name, "sim_thr": thr, "n": int(len(res["y"])), "n_pos": int(res["y"].sum()),
           "n_classI": int((res["df"]["MHC_Class_Used"] == "I").sum())}
    for m, tag in (("weighted_ensemble", "ens"), ("extra_trees", "et")):
        for k_ in ("roc_auc", "pr_auc", "mcc"):
            row[f"{tag}_{k_}"] = float(s.loc[m, f"{k_}_mean"])
        row[f"{tag}_auc_lo"] = float(s.loc[m, "roc_auc_ci95_low"]); row[f"{tag}_auc_hi"] = float(s.loc[m, "roc_auc_ci95_high"])
    train_seqs = res["df"][cfg_s.seq_col].astype(str).str.upper().tolist()
    M = process.cdist(EXT["Peptide"].tolist(), train_seqs, scorer=Levenshtein.normalized_similarity, workers=-1)
    ext = EXT.copy(); ext["max_id_to_train"] = M.max(axis=1)
    ext.to_csv(outdir / "ext_input.csv", index=False)
    pred = predict_new_peptides(str(outdir / "ext_input.csv"), str(outdir / "final_model.pkl"))
    m_ = ext.merge(pred[["dataset_id", "predicted_probability"]], on="dataset_id", how="inner")
    m_["keep"] = m_["max_id_to_train"] < thr
    m_.to_csv(outdir / "ext_pred_merged.csv", index=False)
    sub = m_[m_["keep"]]
    row["ext_all_n"] = int(len(m_)); row["ext_all_auc"] = float(roc_auc_score(m_["label"], m_["predicted_probability"]))
    row["ext_nolap_n"] = int(len(sub))
    a, lo, hi = boot_auc(sub["label"], sub["predicted_probability"])
    row.update(ext_auc=a, ext_auc_lo=lo, ext_auc_hi=hi)
    for c_, tag in (("I", "classI"), ("II", "classII")):
        sc = sub[sub["MHC_Class_Used"] == c_]
        if sc["label"].nunique() == 2:
            a, lo, hi = boot_auc(sc["label"], sc["predicted_probability"])
            row.update({f"ext_{tag}_auc": a, f"ext_{tag}_lo": lo, f"ext_{tag}_hi": hi})
    row["minutes"] = round((time.time() - t0) / 60, 1)
    json.dump(row, open(rowf, "w"), indent=2)
    return row

B_res = {}
B_base_path = Path(BASE_EXT_PRED)
if RUN_DESIGN_B:
    scen_B = {f"sim_{thr:.2f}": thr for thr in B_THRESHOLDS}
    if not B_base_path.exists():
        scen_B = {"sim_0.80": 0.80, **scen_B}
        B_base_path = OUT / "B" / "sim_0.80" / "ext_pred_merged.csv"
    for nm, thr in scen_B.items():
        print("\n" + "=" * 70 + f"\nDESIGN B: {nm}\n" + "=" * 70)
        try:
            B_res[nm] = run_B(nm, thr)
        except Exception:
            traceback.print_exc(); print(f"!!! {nm} failed")

In [ ]:
def paired_delta(base_path, scen_path, n_boot=N_BOOT, seed=7):
    b = pd.read_csv(base_path); b = b[b["keep"]]
    s = pd.read_csv(scen_path); s = s[s["keep"]]
    m = b.merge(s[["dataset_id", "predicted_probability"]], on="dataset_id", suffixes=("_b", "_s"))
    y = m["label"].to_numpy(); pb = m["predicted_probability_b"].to_numpy(); ps = m["predicted_probability_s"].to_numpy()
    rng = np.random.RandomState(seed); dif = []
    for _ in range(n_boot):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) < 2: continue
        dif.append(roc_auc_score(y[i], ps[i]) - roc_auc_score(y[i], pb[i]))
    lo, hi = np.percentile(dif, [2.5, 97.5])
    return roc_auc_score(y, ps) - roc_auc_score(y, pb), lo, hi, len(m)

B_tab = pd.DataFrame(list(B_res.values()))
if len(B_tab):
    base_int = PREV_BASE_AUC if "sim_0.80" not in B_res else B_res["sim_0.80"]["ens_roc_auc"]
    B_tab["d_ens_auc"] = B_tab["ens_roc_auc"] - base_int
    ext_d = []
    for nm in B_tab["scenario"]:
        if nm == "sim_0.80": ext_d.append((0.0, np.nan, np.nan, np.nan)); continue
        ext_d.append(paired_delta(B_base_path, OUT / "B" / nm / "ext_pred_merged.csv"))
    B_tab[["d_ext_auc", "d_ext_lo", "d_ext_hi", "n_common"]] = pd.DataFrame(ext_d, index=B_tab.index)
    B_tab["stable"] = [True if nm == "sim_0.80" else bool(abs(di) <= TOL and lo <= 0 <= hi)
                       for nm, di, lo, hi in zip(B_tab["scenario"], B_tab["d_ens_auc"], B_tab["d_ext_lo"], B_tab["d_ext_hi"])]
    B_tab.to_csv(OUT / "designB_table.csv", index=False)
    colsB = ["scenario", "n", "n_pos", "n_classI", "ens_roc_auc", "ens_auc_lo", "ens_auc_hi", "et_roc_auc", "ext_nolap_n", "ext_auc",
             "ext_auc_lo", "ext_auc_hi", "ext_classI_auc", "ext_classII_auc", "d_ens_auc", "d_ext_auc", "d_ext_lo", "d_ext_hi", "stable"]
    print(B_tab[[c for c in colsB if c in B_tab.columns]].round(3).to_string(index=False))
else:
    print("No Design B results")

## Step 8: Download
The small files (csv, json, png) in `sens_r24/` are zipped into `r24_results.zip` in `BASE_DIR`. Paste them into the chat and I can write the Methods, Results and response letter.

In [ ]:
import zipfile
zp = "r24_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p_ in OUT.rglob("*"):
        if p_.is_file() and p_.suffix in (".csv", ".json", ".png") and p_.stat().st_size < 50e6:
            z.write(p_)
print("saved", Path(zp).resolve())